In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

2025-10-16 18:43:10.448583: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
with open('tokenizer.pkl','rb') as file:
    tokenzer=pickle.load(file)

with open('encoder.pkl','rb') as file:
    encoder=pickle.load(file)

model=load_model('Model_GRU_FAKE_TRUE.keras')

In [3]:
TITLE_MAX_LENGTH = 100
TEXT_MAX_LENGTH = 1500

In [20]:
sample_news_1 = {
        'title': "You Won't Believe Which Celebrity Was Just Replaced By A Robot Clone!",
        'text':"An anonymous insider has leaked documents proving that a beloved Hollywood actor was secretly replaced by an advanced android last month. The source claims the switch was made to hide the actor's controversial political views. This is the scandal of the century, and the mainstream media is refusing to cover it up.",
        'subject': "News" 
    }

In [21]:
import re
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /home/studio-lab-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/studio-lab-
[nltk_data]     user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [22]:
df=pd.DataFrame([sample_news_1])

In [23]:
df

,title,text,subject
0,You Won't Believe Which Celebrity Was Just Rep...,An anonymous insider has leaked documents prov...,News


In [24]:
lemmatizer = WordNetLemmatizer()
def lammatize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    lammatized_words=[lemmatizer.lemmatize(word) for word in words if word not in stopwords.words('english')]
    return " ".join(lammatized_words)

In [25]:
df['text'] = df['text'].apply(lammatize_text)

In [26]:
df['title'] = df['title'].apply(lammatize_text)

In [27]:
def process_text_with_tokenizer(text_series, tokenizer, max_length):
    sequences = tokenizer.texts_to_sequences(text_series.astype(str))

    padded_sequences = pad_sequences(
        sequences,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )
    return padded_sequences

padded_title_sequences = process_text_with_tokenizer(df['title'], tokenzer, TITLE_MAX_LENGTH)
df['title_vectors'] = list(padded_title_sequences)

padded_text_sequences = process_text_with_tokenizer(df['text'], tokenzer, TEXT_MAX_LENGTH)
df['text_vectors'] = list(padded_text_sequences)

In [28]:
print("Model input shapes:")
for i, input_layer in enumerate(model.inputs):
    print(f"Input {i} ({input_layer.name}): {input_layer.shape}")

print("\nModel output shape:")
print(f"Output: {model.output.shape}")

Model input shapes:
Input 0 (title_input): (None, 100)
Input 1 (text_input): (None, 1500)
Input 2 (other_features_input): (None, 7)

Model output shape:
Output: (None, 1)


In [29]:
subject_encoded = encoder.transform(df[['subject']])

In [30]:
df_other_features = subject_encoded.astype('float32')

In [31]:
df_title = np.stack(df['title_vectors'].values)
df_text = np.stack(df['text_vectors'].values)

In [32]:
predictions_proba = model.predict(
    [df_title,df_text, df_other_features],
    verbose=1
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step


In [33]:
if predictions_proba>=.6:
    print('True News')
else:
    print('Fake News')

Fake News
